### Importing Libraries

In [2]:
# ─────────────────────────────────────────────────────────────
# 0.  STANDARD LIBRARY
# ─────────────────────────────────────────────────────────────
import os
import sys
import json
import random
import logging
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 1.  THIRD-PARTY — NUMERIC / DATA
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# 2.  COMPUTER VISION & AUDIO
# ─────────────────────────────────────────────────────────────
import cv2
import librosa
import librosa.display

# ─────────────────────────────────────────────────────────────
# 3.  MACHINE LEARNING — SKLEARN
# ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────────────────────
# 4.  DEEP LEARNING — TENSORFLOW / KERAS
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import Model, Input, regularizers
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, GlobalAveragePooling2D,
    GaussianNoise, Activation, Concatenate, Add
)
from tensorflow.keras.applications import DenseNet121, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.constraints import MaxNorm

# ─────────────────────────────────────────────────────────────
# 5.  VISUALISATION
# ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────────────────────
# 6.  SUPPRESS NOISE & REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

### Environment Setup

In [53]:
MASTER_CLASSES = ["asthma", "copd", "healthy", "pneumonia"]
MASTER_LE = LabelEncoder()
MASTER_LE.fit(MASTER_CLASSES)

NUM_CLASSES    = len(MASTER_CLASSES)
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
EPOCHS         = 50

BASE_DIR   = Path("respiratory_global_pipeline")
DATA_DIR   = BASE_DIR / "data"
MODEL_DIR  = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

In [4]:
for d in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TABULAR_FEATURES = [
    "age", "gender_enc", "smoking_status",
    "fev1_percent", "spo2", "respiratory_rate",
    "cough_severity", "wheeze", "chest_tightness",
    "crackles", "fever", "bmi", "copd_exacerbations",
]
CLASSES_TO_DROP = ["bronchitis", "lung_cancer"]
FUSION_WEIGHTS = {"image": 0.50, "tabular": 0.35, "audio": 0.15}

print(f"  Master classes : {MASTER_CLASSES}")
print(f"  Label encoding : {dict(zip(MASTER_CLASSES, MASTER_LE.transform(MASTER_CLASSES)))}")

  Master classes : ['asthma', 'copd', 'healthy', 'pneumonia']
  Label encoding : {'asthma': np.int64(0), 'copd': np.int64(1), 'healthy': np.int64(2), 'pneumonia': np.int64(3)}


### Importing Dataset from kaggle

In [5]:
if not os.path.exists('/root/.kaggle'):
    os.makedirs('/root/.kaggle')

!mv kaggle.json /root/.kaggle/ 2>/dev/null || true
!chmod 600 /root/.kaggle/kaggle.json 2>/dev/null || true

#### Image Dataset

In [6]:
nih_dir = DATA_DIR / "nih_chestxray"
nih_dir.mkdir(parents=True, exist_ok=True)

if not any(nih_dir.iterdir()):
    print("[Kaggle] Downloading NIH dataset...")
    os.system(f"kaggle datasets download -d khanfashee/nih-chest-x-ray-14-224x224-resized -p {nih_dir} --unzip")

[Kaggle] Downloading NIH dataset...


#### Audio Dataset

In [7]:
icbhi_dir = DATA_DIR / "icbhi"
icbhi_dir.mkdir(parents=True, exist_ok=True)

if not any(icbhi_dir.iterdir()):
    print("[Kaggle] Downloading ICBHI dataset...")
    os.system(f"kaggle datasets download -d vbookshelf/respiratory-sound-database -p {icbhi_dir} --unzip")

[Kaggle] Downloading ICBHI dataset...


#### Tabular Dataset

In [8]:
tabular_csv_path = Path('/content/clinical_data.csv')

if not tabular_csv_path.exists():
    print(f"[Warning] Tabular CSV not found at {tabular_csv_path}. Please upload it before running the pipeline.")

### Preprocessing on Image dataset

In [46]:
NIH_LABEL_MAP = {
    "Emphysema": "copd",
    "Pneumonia": "pneumonia",
    "Atelectasis": "asthma",
    "No Finding": "healthy"
}

In [47]:
def load_and_filter_nih_metadata(image_dir: Path, healthy_limit: int = 2000) -> pd.DataFrame:
    """Pairs metadata with paths and BALANCES classes for speed."""
    meta_path = next(image_dir.rglob("Data_Entry_2017*.csv"), None)
    if meta_path is None: return pd.DataFrame(columns=["filepath", "label", "label_enc"])

    df = pd.read_csv(meta_path)
    df = df[df["Finding Labels"].isin(NIH_LABEL_MAP.keys())].copy()
    df["label"] = df["Finding Labels"].map(NIH_LABEL_MAP).str.lower()

    path_lookup = {p.name: str(p) for p in image_dir.rglob("*.png")}
    df["filepath"] = df["Image Index"].map(path_lookup)
    df = df.dropna(subset=["filepath"])

    healthy_df  = df[df["label"] == "healthy"]
    diseased_df = df[df["label"] != "healthy"]

    if len(healthy_df) > healthy_limit:
        healthy_df = healthy_df.sample(n=healthy_limit, random_state=SEED)

    df = pd.concat([healthy_df, diseased_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    df["label_enc"] = MASTER_LE.transform(df["label"])

    print(f"\n[Image] BALANCED NIH subset: {len(df)} rows")
    print(df["label"].value_counts().to_string())

    return df[["filepath", "label", "label_enc"]]

In [48]:
def preprocess_chest_xray(image_path: str, target_size: tuple = IMG_SIZE) -> np.ndarray:
    img_gray  = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None: raise ValueError(f"Cannot load: {image_path}")

    clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)

    img_rgb   = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
    img_res   = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_AREA)

    return (img_res / 255.0).astype(np.float32)

In [49]:
class NIHImageGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, augment=False, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

        self.aug = ImageDataGenerator(
            rotation_range=10, width_shift_range=0.08,
            height_shift_range=0.08, zoom_range=0.08, horizontal_flip=True,
        )

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]
        X, y = [], []

        for _, row in batch_df.iterrows():
            try: img = preprocess_chest_xray(row["filepath"])
            except: img = np.zeros((*IMG_SIZE, 3), dtype=np.float32)
            X.append(img)
            y.append(row["label_enc"])

        X = np.array(X, dtype=np.float32)
        y = to_categorical(np.array(y), num_classes=NUM_CLASSES)

        if self.augment:
            X = next(self.aug.flow(X, batch_size=len(X), shuffle=False))
        return X, y

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

### Built Image Model

In [50]:
def build_image_model(learning_rate: float = 1e-4) -> Model:
    base = DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True

    inputs  = Input(shape=(*IMG_SIZE, 3), name="image_input")
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs, name="DenseNet121_ChestXray")
    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-4),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [51]:
def train_image_model(image_dir: Path):
    print("\n" + "-" * 60 + "\n  IMAGE BRANCH - Strategy B (Balanced Training)\n" + "-" * 60)

    df = load_and_filter_nih_metadata(image_dir)
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    y_train = train_df["label_enc"].values
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw_dict = {int(c): w for c, w in zip(np.unique(y_train), cw)}

    train_gen = NIHImageGenerator(train_df, augment=True,  shuffle=True)
    val_gen   = NIHImageGenerator(val_df,   augment=False, shuffle=False)
    test_gen  = NIHImageGenerator(test_df,  augment=False, shuffle=False)

    model = build_image_model()
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_image_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                        callbacks=callbacks, class_weight=cw_dict, verbose=1)

    # Eval
    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print("\n[Image Branch Evaluation]")
    print(classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=MASTER_CLASSES))

    plot_confusion_matrix(y_true, y_pred, "Image_Expert")
    model.save(str(MODEL_DIR / "final_image_model.keras"))
    return model, {"history": history.history}

### Preprocessing on Audio

In [15]:
ICBHI_LABEL_MAP = {
    "COPD": "copd",
    "Pneumonia": "pneumonia",
    "Asthma": "asthma",
    "Healthy": "healthy"
}

In [16]:
def load_icbhi_metadata(audio_dir: Path) -> pd.DataFrame:
    diag_path = next(audio_dir.rglob("patient_diagnosis.csv"), None)
    if diag_path is None: return pd.DataFrame(columns=["filepath", "label", "label_enc"])

    diag_df = pd.read_csv(diag_path, header=None, names=["patient_id", "diagnosis"])
    diag_df = diag_df[diag_df["diagnosis"].isin(ICBHI_LABEL_MAP.keys())].copy()
    diag_df["label"] = diag_df["diagnosis"].map(ICBHI_LABEL_MAP)

    records = []

    for wav in audio_dir.rglob("*.wav"):
        try:
            pid = int(wav.stem.split("_")[0])
            row = diag_df[diag_df["patient_id"] == pid]
            if not row.empty:
                records.append({"filepath": str(wav), "patient_id": pid, "label": row.iloc[0]["label"]})
        except (ValueError, IndexError):
            continue

    df = pd.DataFrame(records).drop_duplicates(subset="filepath")

    if not df.empty:
        df["label_enc"] = MASTER_LE.transform(df["label"])

        print(f"\n[Audio] ICBHI subset : {len(df)} rows")
        print(df["label"].value_counts().to_string())
    else:
        print("[Error] No audio files matched the diagnosis CSV.")

    return df

In [17]:
def wav_to_melspectrogram(filepath: str, sr: int = 22050, n_mels: int = 128, fmax: int = 2000, duration: float = 5.0, target_size: tuple = IMG_SIZE) -> np.ndarray:
    try:
        y, _ = librosa.load(filepath, sr=sr, duration=duration, mono=True)
    except Exception:
        return np.zeros((*target_size, 3), dtype=np.float32)

    target_len = int(sr * duration)
    y = np.pad(y, (0, max(0, target_len - len(y))), mode="constant")[:target_len]

    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_n  = ((mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8) * 255).astype(np.uint8)
    mel_r  = cv2.resize(mel_n, target_size, interpolation=cv2.INTER_LINEAR)

    return np.stack([mel_r] * 3, axis=-1).astype(np.float32) / 255.0

In [18]:
class ICBHIAudioGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]

        X = np.array([wav_to_melspectrogram(r["filepath"]) for _, r in batch_df.iterrows()], dtype=np.float32)
        y = to_categorical(batch_df["label_enc"].values, num_classes=NUM_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

### Built Model on Audio data

In [19]:
def build_audio_model(learning_rate: float = 5e-4) -> Model:
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False

    inputs  = Input(shape=(*IMG_SIZE, 3), name="audio_input")
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs, name="EfficientNetB0_Audio")

    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-4),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [20]:
def train_audio_model(audio_dir: Path):
    print("Training Audio Model...")

    df = load_icbhi_metadata(audio_dir)

    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    train_gen = ICBHIAudioGenerator(train_df, shuffle=True)
    val_gen   = ICBHIAudioGenerator(val_df,   shuffle=False)
    test_gen  = ICBHIAudioGenerator(test_df,  shuffle=False)

    model = build_audio_model()
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_audio_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print("\n[Audio Branch Evaluation]")
    print(classification_report(y_true, y_pred,
                                labels=np.arange(NUM_CLASSES),
                                target_names=MASTER_CLASSES))

    plot_confusion_matrix(y_true, y_pred, "Audio_Standalone")

    f1 = f1_score(y_true, y_pred, average="macro")
    model.save(str(MODEL_DIR / "final_audio_model.keras"))

    return model, {"history": history.history, "test_f1": f1}

### Preprocessing on Tabular dataset

In [21]:
def load_and_prepare_tabular(csv_path: Path):
    df = pd.read_csv(csv_path)

    df = df[~df["diagnosis"].isin(CLASSES_TO_DROP)].copy()
    df["diagnosis"] = df["diagnosis"].str.lower().str.strip()
    df = df[df["diagnosis"].isin(MASTER_CLASSES)].copy()

    df["gender_enc"] = (df["gender"].str.upper() == "M").astype(int)
    df["label_enc"]  = MASTER_LE.transform(df["diagnosis"])

    print(f"\n[Tabular] Total records loaded: {len(df)}")
    print(df["diagnosis"].value_counts().to_string())

    X = df[TABULAR_FEATURES].values.astype(np.float32)
    y = df["label_enc"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val   = scaler.transform(X_val).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler

In [22]:
def mixup_tabular(X: np.ndarray, y: np.ndarray, alpha: float = 0.3, n_synthetic: int = 1200) -> tuple:
    rng     = np.random.default_rng(SEED)
    n       = len(X)
    lambdas = rng.beta(alpha, alpha, size=n_synthetic)
    idx_a   = rng.integers(0, n, size=n_synthetic)
    idx_b   = rng.integers(0, n, size=n_synthetic)
    lam     = lambdas[:, None]

    X_mix = lam * X[idx_a] + (1 - lam) * X[idx_b]
    oh_a  = to_categorical(y[idx_a], num_classes=NUM_CLASSES)
    oh_b  = to_categorical(y[idx_b], num_classes=NUM_CLASSES)
    y_mix = np.argmax(lambdas[:, None] * oh_a + (1 - lambdas[:, None]) * oh_b, axis=1)

    return (np.vstack([X, X_mix]).astype(np.float32), np.concatenate([y, y_mix]))

In [23]:
def residual_mlp_block(x, units: int, dropout_rate: float, l2_reg: float = 1e-4):
    reg    = regularizers.l2(l2_reg)
    cnstr  = MaxNorm(max_value=3.0)
    skip   = x
    x = Dense(units, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(units, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip])
    x = Activation("gelu")(x)

    return x

### Built Model On Tabular dataset

In [24]:
def build_tabular_dnn(n_features: int, lr_schedule) -> Model:
    reg   = regularizers.l2(1e-4)
    cnstr = MaxNorm(max_value=3.0)

    inputs = Input(shape=(n_features,), name="tabular_input")

    x = GaussianNoise(0.02)(inputs)

    x = Dense(256, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False, name="proj_256")(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)

    x = residual_mlp_block(x, units=256, dropout_rate=0.40)

    skip_128 = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    skip_128 = BatchNormalization()(skip_128)
    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.35)(x)
    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip_128])
    x = Activation("gelu")(x)

    skip_64 = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    skip_64 = BatchNormalization()(skip_64)
    x = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.25)(x)
    x = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip_64])
    x = Activation("gelu")(x)

    x = Dense(32, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False, name="bottleneck")(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.20)(x)

    outputs = Dense(NUM_CLASSES, activation="softmax", name="class_probs")(x)

    model = Model(inputs, outputs, name="Residual_TabularDNN")
    model.compile(optimizer=AdamW(learning_rate=lr_schedule, weight_decay=5e-4),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    return model

In [25]:
def train_tabular_model(csv_path: Path):
    print("Training Tabular Model...")

    (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler = load_and_prepare_tabular(csv_path)
    X_train_aug, y_train_aug = mixup_tabular(X_train, y_train, alpha=0.3, n_synthetic=1200)

    cls_weights = compute_class_weight("balanced", classes=np.unique(y_train_aug), y=y_train_aug)
    class_weight_dict = {i: float(w) for i, w in enumerate(cls_weights)}

    steps_per_epoch = int(np.ceil(len(X_train_aug) / BATCH_SIZE))
    lr_schedule = CosineDecayRestarts(initial_learning_rate=5e-4,
                                     first_decay_steps=steps_per_epoch * 10,
                                     t_mul=2.0, m_mul=0.9, alpha=1e-6)

    model = build_tabular_dnn(n_features=len(TABULAR_FEATURES), lr_schedule=lr_schedule)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_tabular_dnn.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(X_train_aug, y_train_aug,
                        validation_data=(X_val, y_val),
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        class_weight=class_weight_dict,
                        callbacks=callbacks,
                        verbose=1)

    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print("\n[Tabular Branch Evaluation]")
    print(classification_report(y_test, y_pred,
                                labels=np.arange(NUM_CLASSES),
                                target_names=MASTER_CLASSES))

    plot_confusion_matrix(y_test, y_pred, "Tabular_Standalone")

    f1 = f1_score(y_test, y_pred, average="macro")
    model.save(str(MODEL_DIR / "final_tabular_dnn.keras"))

    return model, f1, scaler

In [26]:
def predict_tabular_probs(model: Model, scaler: StandardScaler, clinical_features: dict) -> np.ndarray:
    x_raw  = np.array([[clinical_features.get(f, 0.0) for f in TABULAR_FEATURES]], dtype=np.float32)
    x_norm = scaler.transform(x_raw).astype(np.float32)
    return model.predict(x_norm, verbose=0)[0]

### Multimodal Data Alignment

In [27]:
def align_multimodal_datasets(image_dir: Path, audio_dir: Path, tabular_csv: Path) -> pd.DataFrame:
    img_df = load_and_filter_nih_metadata(image_dir)
    aud_df = load_icbhi_metadata(audio_dir)

    tab_df = pd.read_csv(tabular_csv)

    tab_df["label"] = tab_df["diagnosis"].str.lower().str.strip()
    tab_df = tab_df[tab_df["label"].isin(MASTER_CLASSES)].copy()
    tab_df["gender_enc"] = (tab_df["gender"].str.upper() == "M").astype(int)

    unified_records = []
    class_counts = {}

    for cls in MASTER_CLASSES:
        imgs = img_df[img_df["label"] == cls]["filepath"].values
        auds = aud_df[aud_df["label"] == cls]["filepath"].values
        tabs = tab_df[tab_df["label"] == cls].to_dict('records')

        min_count = min(len(imgs), len(auds), len(tabs))
        class_counts[cls] = min_count

        np.random.shuffle(imgs)
        np.random.shuffle(auds)
        np.random.shuffle(tabs)

        for i in range(min_count):
            record = tabs[i].copy()
            record["img_path"] = imgs[i]
            record["audio_path"] = auds[i]
            unified_records.append(record)

    unified_df = pd.DataFrame(unified_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
    unified_df["label_enc"] = MASTER_LE.transform(unified_df["label"])


    for cls, count in class_counts.items():
        print(f"  {cls.upper():<10} : {count} triplets")

    print(f"\n  Total Triplets: {len(unified_df)}")
    print("="*30)

    return unified_df

In [28]:
class GlobalMultimodalGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, tabular_scaler, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.scaler = tabular_scaler
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]

        X_img, X_aud, X_tab, y = [], [], [], []
        for _, row in batch_df.iterrows():
            try: X_img.append(preprocess_chest_xray(row["img_path"]))
            except: X_img.append(np.zeros((*IMG_SIZE, 3), dtype=np.float32))

            X_aud.append(wav_to_melspectrogram(row["audio_path"]))

            X_tab.append(row[TABULAR_FEATURES].values.astype(np.float32))

            y.append(row["label_enc"])

        X_tab_scaled = self.scaler.transform(np.array(X_tab))

        X_dict = {
            "image_input":   np.array(X_img, dtype=np.float32),
            "audio_input":   np.array(X_aud, dtype=np.float32),
            "tabular_input": X_tab_scaled.astype(np.float32)
        }

        y_cat = to_categorical(np.array(y), num_classes=NUM_CLASSES)
        return X_dict, y_cat

### Global (Joint) Model Architecture

In [29]:
def build_global_model(img_branch, aud_branch, tab_branch, lr_schedule) -> Model:
    reg = regularizers.l2(1e-4)
    cnstr = MaxNorm(max_value=3.0)

    # 1. Extract the feature vectors (The Dropout layer before Softmax)
    img_feat = img_branch.layers[-2].output
    aud_feat = aud_branch.layers[-2].output
    tab_feat = tab_branch.layers[-2].output

    merged = Concatenate(name="fusion_concat")([img_feat, aud_feat, tab_feat])

    # 3. Joint Reasoning Layers
    x = Dense(256, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(merged)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.40)(x)

    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.30)(x)

    # Single unified prediction head (Automatically 4 units)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="final_prediction")(x)

    # 4. Build Global Model
    model = Model(
        inputs=[img_branch.input, aud_branch.input, tab_branch.input],
        outputs=outputs,
        name="Global_Multimodal_Net"
    )

    model.compile(
        optimizer=AdamW(learning_rate=lr_schedule, weight_decay=5e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

### Plotting and Report Tools

In [30]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> None:
    cm     = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    labels = MASTER_LE.classes_

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)

    ax.set_title(f"Confusion Matrix - {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()

    safe = title.replace(" ", "_").replace("(", "").replace(")", "")
    path = REPORT_DIR / f"cm_{safe}.png"
    fig.savefig(str(path), dpi=150)
    plt.close(fig)

    print(f"[Plot] Saved confusion matrix to: {path}")

In [31]:
def plot_training_history(history: dict, model_name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history["loss"], label="Train Loss", linewidth=2)
    axes[0].plot(history["val_loss"], label="Val Loss", linewidth=2, linestyle="--")
    axes[0].set_title(f"{model_name} - Loss"); axes[0].legend()
    axes[1].plot(history["accuracy"], label="Train Acc", linewidth=2)
    axes[1].plot(history["val_accuracy"], label="Val Acc", linewidth=2, linestyle="--")
    axes[1].set_title(f"{model_name} - Accuracy"); axes[1].legend()
    plt.tight_layout()
    path = REPORT_DIR / f"history_{model_name.replace(' ', '_')}.png"
    fig.savefig(str(path), dpi=150)
    plt.close(fig)

In [32]:
def build_joint_doctor_report(probs: np.ndarray, patient_meta: dict) -> dict:
    idx = int(np.argmax(probs))
    final_class = MASTER_LE.inverse_transform([idx])[0]
    conf_pct = float(probs[idx]) * 100

    recommendations = {
        "copd":      ["Spirometry (FEV1/FVC < 0.70)", "CT chest for emphysema", "Initiate LABA/LAMA"],
        "pneumonia": ["Chest X-ray for consolidation", "CBC, CRP monitoring", "Empiric antibiotics"],
        "asthma":    ["Peak flow monitoring", "IgE testing", "Initiate ICS step therapy"],
        "healthy":   ["No acute respiratory pathology detected", "Annual routine check-up", "Maintain healthy lifestyle"]
    }

    return {
        "report_metadata": {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "pipeline_type": "Joint/Early Fusion",
            "patient_info": patient_meta,
        },
        "joint_diagnosis": {
            "final_class": final_class,
            "confidence_percent": round(conf_pct, 2),
            "probabilities": {cls: round(float(p)*100, 2) for cls, p in zip(MASTER_CLASSES, probs)},
        },
        "clinical_flags": {
            "high_uncertainty": conf_pct < 65.0,
            "human_review_needed": conf_pct < 65.0,
        },
        "clinical_recommendations": recommendations.get(final_class, []),
    }

In [33]:
def print_joint_report(report: dict) -> None:
    jd = report["joint_diagnosis"]
    cf = report["clinical_flags"]

    print("\n" + "=" * 60)
    print(f"  DOCTOR REPORT - Patient {report['report_metadata']['patient_info'].get('patient_id', 'Unknown')}")
    print("=" * 60)
    print(f"  Final Diagnosis      : {jd['final_class'].upper()}")
    print(f"  Confidence           : {jd['confidence_percent']:.1f}%")
    print(f"  High Uncertainty     : {cf['high_uncertainty']}")
    print(f"  Human Review Needed  : {cf['human_review_needed']}")
    print("\n  Joint Softmax Probabilities:")

    for cls, prob in jd["probabilities"].items():
        bar = "#" * int(prob / 5)
        print(f"    {cls:<12} {prob:5.1f}%  {bar}")

    print("\n  Clinical Recommendations:")

    for rec in report["clinical_recommendations"]:
        print(f"    - {rec}")

    print("=" * 60)

### Global Model Training Flow & Orchestrator

In [34]:
def split_joint_data(df: pd.DataFrame):
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    return train_df, val_df, test_df

In [35]:
def get_multimodal_generators(train_df, val_df, test_df, scaler):
    train_gen = GlobalMultimodalGenerator(train_df, scaler, shuffle=True)
    val_gen   = GlobalMultimodalGenerator(val_df, scaler, shuffle=False)
    test_gen  = GlobalMultimodalGenerator(test_df, scaler, shuffle=False)

    return train_gen, val_gen, test_gen

In [36]:
def instantiate_branches(lr_schedule):
    print("[Setup] Loading Expert weights into modular branches...")

    try:
        img_branch = tf.keras.models.load_model(str(MODEL_DIR / "best_image_model.keras"))
        aud_branch = tf.keras.models.load_model(str(MODEL_DIR / "best_audio_model.keras"))
        tab_branch = tf.keras.models.load_model(str(MODEL_DIR / "best_tabular_dnn.keras"))

        # Lock them so we only train the new fusion layers first
        img_branch.trainable = False
        aud_branch.trainable = False
        tab_branch.trainable = False
        print("  Successfully loaded and locked all 3 experts.")

    except Exception as e:
        print(f"  [Warning] Experts not found: {e}. Building fresh.")
        img_branch = build_image_model()
        aud_branch = build_audio_model()
        tab_branch = build_tabular_dnn(n_features=len(TABULAR_FEATURES), lr_schedule=lr_schedule)

    return img_branch, aud_branch, tab_branch

In [37]:
def train_global_model_core(train_gen, val_gen, branches, steps_per_epoch):
    img_b, aud_b, tab_b = branches
    lr_schedule = CosineDecayRestarts(initial_learning_rate=3e-4, first_decay_steps=steps_per_epoch * 10, t_mul=2.0, m_mul=0.9, alpha=1e-6)
    model = build_global_model(img_b, aud_b, tab_b, lr_schedule)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_global_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    return model, history

### Evaluate Global Model

In [38]:
def evaluate_global_model(model, test_gen, test_df):
    print("\n[Evaluation] Testing Global Model on hold-out test set...")

    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print(classification_report(y_true, y_pred,
                                labels=np.arange(NUM_CLASSES),
                                target_names=MASTER_CLASSES))

    plot_confusion_matrix(y_true, y_pred, "Global_Model_4Class")
    return y_prob

In [39]:
def run_inference_demo(model, scaler, demo_row):
    X_dict = {
        "image_input":   np.expand_dims(preprocess_chest_xray(demo_row["img_path"]), axis=0),
        "audio_input":   np.expand_dims(wav_to_melspectrogram(demo_row["audio_path"]), axis=0),
        "tabular_input": scaler.transform([demo_row[TABULAR_FEATURES].values.astype(np.float32)])
    }

    probs = model.predict(X_dict, verbose=0)[0]

    report = build_joint_doctor_report(probs, patient_meta={
        "patient_id": demo_row.get("patient_id", "Demo-001"),
        "age": demo_row.get("age", 0)
    })

    print_joint_report(report)

### Main function

In [40]:
def training_phase():
    print("\n" + "="*70 + "\n  PHASE 1: INDEPENDENT EXPERT TRAINING\n" + "="*70)

    train_image_model(nih_dir)
    train_audio_model(icbhi_dir)

    _, _, expert_scaler = train_tabular_model(tabular_csv_path)

    return expert_scaler

In [41]:
def run_global_pipeline():
    # Step 1: Get the 'baton' (scaler) from Phase 1
    expert_scaler = training_phase()

    if not tabular_csv_path.exists():
        print(f"[Error] CSV not found."); return

    # Step 2: Alignment
    unified_df = align_multimodal_datasets(nih_dir, icbhi_dir, tabular_csv_path)

    # Step 3: Use SRP helpers to prep data (Consistent scaling guaranteed)
    train_df, val_df, test_df = split_joint_data(unified_df)
    train_gen, val_gen, test_gen = get_multimodal_generators(train_df, val_df, test_df, expert_scaler)

    # Step 4: Setup & Train
    branches = instantiate_branches(lr_schedule=1e-5)
    steps = int(np.ceil(len(train_df) / BATCH_SIZE))
    global_model, history = train_global_model_core(train_gen, val_gen, branches, steps)

    # Step 5: Report & Demo
    evaluate_global_model(global_model, test_gen, test_df)
    plot_training_history(history.history, "Global_Strategy_B")
    run_inference_demo(global_model, expert_scaler, test_df.sample(1).iloc[0])

In [54]:
if __name__ == "__main__":
    run_global_pipeline()


  PHASE 1: INDEPENDENT EXPERT TRAINING

------------------------------------------------------------
  IMAGE BRANCH - Strategy B (Balanced Training)
------------------------------------------------------------

[Image] BALANCED NIH subset: 7414 rows
label
asthma       4212
healthy      2000
copd          895
pneumonia     307
Epoch 1/50
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 0.2569 - loss: 1.7512
Epoch 1: val_loss improved from None to 1.29058, saving model to respiratory_global_pipeline/models/best_image_model.keras

Epoch 1: finished saving model to respiratory_global_pipeline/models/best_image_model.keras
158/158 ━━━━━━━━━━━━━━━━━━━━ 133s 651ms/step - accuracy: 0.2720 - loss: 1.7471 - val_accuracy: 0.3989 - val_loss: 1.2906 - learning_rate: 1.0000e-04
Epoch 2/50
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step - accuracy: 0.3164 - loss: 1.5648
Epoch 2: val_loss improved from 1.29058 to 1.25841, saving model to respiratory_global_pipeline/models/best_image_model.keras

Epo